# 파이썬에서 자료구조 만들기
> 해당 작업은 100% 과제를 위한 것입니다.

# 3가지 자료구조
자료구조를 배우게되면 가장 먼저 배우는 3가지가 존재합니다. (Map, Set, Graph 등도 있지만 지금은 과제 흐름데로...)
- 배열
- 큐
- 스택

우선 순서대로 설명을 먼저 한 뒤, 파이썬에 알맞는 구현방식을 이용하여 자료구조를 생성해보겠습니다.

## 배열
배열은 기본적으로 연속된 메모리에 데이터를 저장하는 자료구조를 말합니다.

말로 설명하면 어려우니 피그마로 표현해보겠습니다.

![text](image-1.png)

그리고 각 메모리는 대체로 `0x0008`, `0x0010`, `0x0018`, `...` 형태로 16진법으로 나타내는 형태가 많습니다.

### 파이썬의 리스트
파이썬은 조금 더 형태가 독특하며, 객체 안에 메타데이터를 이것저것 가지고 있습니다.
- refcnt: 참조하고 있는 변수 개수
- type: 타입 정보(list)
- size: 원소 개수
- allocated: 미리 확보한 공간 (계단식 점프 증가)
- `__iter__()` 메서드가 존재
- `sequence`, `mutable` 특징이 존재합니다.

In [ ]:
# 확인용 스크립트

l = [10, 20, 30]

it = iter(l)
print(it)
print(f"hex(id(l)): {hex(id(l))}")
print(f"hex(id(l)): {hex(id(it))}")
try:
    while True:
        n = next(it)
        print(n)
except StopIteration as e:
    print("StopIteration Occured")

    

hex(id(l)): 0x1af4d60f300
hex(id(l)): 0x1af4d6a7460
10
20
30
StopIteration Occured


In [35]:
# 구현하기
# 고려할 것: next(), append(), __delitem__() 정도만 만들고 은닉까지 해주기
class MyList:
    # 인덱스 접근은 O(1)인데 파이썬으로 어떻게 파이썬 list보다 더 빨리 접근하지?
    # 생각나는건 dict밖에 없는데
    def __init__(self, start):
        self.__space = {}
        self.__le = 0

        for i, v in enumerate(iter(start)):
            self.__space[i] = v
            self.__le += 1

    # 추가와 삭제.
    # 추가는 append()
    # 삭제는 del item
##############################################################################
    def append(self, val):
        # 이거 실제 방식하고 좀 다르긴 한데 일단 추가는 되어야하니까.
        # 동적 배열을 구현하는건 어려울듯...
        # ctype 사용하면 되지만 다른 작업 할게 있어서 간소히 구현하겠습니다.
        self.__space[self.le] = val
        self.__le += 1
    

    # del myList[i]로 삭제
    def __delitem__(self, index):
        if not isinstance(index, int):
            print("int 여야함")
            print(f"index: {type(index)}")
            return
        if index >= self.le:
            print(f"너무 큰 인덱스. length = {self.le}")
            return
        if index < 0:
            print("제대로된 인덱스점...")
            return
        for k, v in self.__space.items():

            if k < index:
                continue
            self.__space[k] = self.__space.get(k+1, None)

        del self.__space[self.__le-1]
        self.__le -= 1

    def __getitem__(self, index):
        r = self.__space.get(index, None)
        return r
##############################################################################

    # length는 조작되면 안되기 때문에 데코레이터를 통한 setter, getter 설정
##############################################################################
    @property
    def le(self):
        return self.__le
    
    @le.setter
    def le(self, val):
        print("length 조작 금지")
##############################################################################

    # for문 등 사용을 할 때 정상적인 장동을 위한 iter 매직 메서드
##############################################################################
    def __iter__(self):
        for i in range(self.__le):
            yield self.__space[i]
##############################################################################

    # 기타 유틸용
    def __repr__(self):
        return str([self.__space[i] for i in range(self.le)])
    

l = MyList([1, 2, 3, 10])
print(l)

for i in l:
    print(i)
print(f"\n --- \n")

l.append(1)

for i in l:
    print(i)

print(f"\n --- \n")

del l[2]

for i in l:
    print(i)

print(f"length: {l.le}")
print("조작 시도 l.le = 10")
l.le = 10

print(l[3])
print(l[0.1])
print(l[10])

[1, 2, 3, 10]
1
2
3
10

 --- 

1
2
3
10
1

 --- 

1
2
10
1
length: 4
조작 시도 l.le = 10
length 조작 금지
1
None
None


# 파이썬의 스택과 큐
## 스택
**스택**은 말 그대로 쌓다 라는 의미를 가진 자료구조이며 **LIFO**(Last In First Out)원칙을 지킵니다. 마지막에 들어간 값이 먼저 나오는 형태입니다.

주요 메서드로는 값을 넣는 `push()`와 가장 최근에 넣은 값을 꺼내는 `pop()`이 있습니다.

보통 파이썬에서는 이를 `list`를 통해 사용합니다.

주요 사용처에는 프로그램의 가장 최근에 사용한 새로운 데이터부터 삭제를 하는 Stack 메모리 영역이나 브라우저같은 최근 기록을 기준으로 쌓을 경우 Stack자료구조의 특징을 사용합니다. 또한 알고리즘의 DFS을 구현하는데 쓰이기도 합니다.

## 큐
**큐**는 스택과 반대로 **FIFO**(First In First Out)원칙을 지킵니다. 먼저 들어간 먼저 나옵니다.

주요 메서드로는 값을 넣는 `enqueue()`와 값을 앞에서 꺼내는 `dequeue()`가 존재합니다.

큐같은 경우에는 자료가 순서대로 들어오고 나가야할 때 사용하며 이는 대체로 순서와 관련된 작업에서 나타납니다. 예를 들면 CPU의 스케줄링 등에 쓰입니다. 이는 알고리즘의 BFS에서 사용됩니다.

> **deque**라는 자료구조 또한 있으며 여기에서는 `append()`, `appendleft()`, `pop()`, `popleft()`가 존재합니다.

In [ ]:
# 빠른 구현
class MyQueue:
    def __init__(self):
        self.__space = []
    
    def enqueue(self, val):
        
        self.__space.append(val)

    def dequeue(self):
        if len(self.__space) == 0:
            return None
        r = self.__space[0]
        self.__space = self.__space[1:]
        return r

    def __repr__(self):
        return str(self.__space)

class MyStack:
    def __init__(self):
        self.__space = []
    
    def push(self, val):
        self.__space.append(val)
    
    def pop(self):
        return self.__space.pop()
    
    def __repr__(self):
        return str(self.__space)

# 테스트 코드


q = MyQueue()
q.enqueue(10)
q.enqueue(20)
q.enqueue(30)
assert q.dequeue() == 10
assert q.dequeue() == 20
assert q.dequeue() == 30
assert q.dequeue() is None

print("MyQueue 정상")

s = MyStack()
s.push(10)
s.push(20)
s.push(30)
assert s.pop() == 30
assert s.pop() == 20
assert s.pop() == 10

print("MyStack 정상")

MyQueue 정상
MyStack 정상
